[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/evaluate_code_readability_classifications.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">代码可读性评估</h1>

Arize 提供评估 LLM 应用的工具，其中包括判断 LLM 应用生成的代码是否可读的工具。

本 Notebook 的目的：

- 使用带有真实标签的数据集，评估借助 LLM 将生成代码分类为可读或不可读的方法表现。
- 提供实验框架，帮助用户迭代并改进默认分类模板。

## 安装依赖并导入库

In [ ]:
#####################
## N_EVAL_SAMPLE_SIZE
#####################
# 评估样本量决定运行时间
# 100 个样本：GPT-4 约 80 秒 / GPT-3.5 约 40 秒
# 1,000 个样本：GPT-4 约 15～17 分钟 / GPT-3.5 约 6～7 分钟，取决于重试
# 10,000 个样本：GPT-4 约 170 分钟 / GPT-3.5 约 70 分钟
N_EVAL_SAMPLE_SIZE = 10

In [ ]:
%pip install -qq "arize-phoenix-evals>=0.0.5" "openai>=1" ipython matplotlib pycm scikit-learn tiktoken nest_asyncio

ℹ️ 在 Jupyter 或 Google Colab 等 Notebook 环境中，可选用 `nest_asyncio` 启用异步请求提交。`nest_asyncio` 会全局修补 `asyncio`，使事件循环可重入。非 Notebook 环境无需此设置。

不使用 `nest_asyncio` 时，评估提交可能慢很多，具体取决于组织的速率限制。通常可提速约 5 倍。

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
from getpass import getpass

import matplotlib.pyplot as plt
import openai
import pandas as pd
from phoenix.evals import (
    ClassificationEvaluator,
    async_evaluate_dataframe,
    download_benchmark_dataset,
)
from phoenix.evals.llm import LLM
from pycm import ConfusionMatrix
from sklearn.metrics import classification_report

pd.set_option("display.max_colwidth", None)

## 下载基准数据集

我们将使用带真实标签的可读与不可读代码基准数据集，评估由 LLM 模型、设置和评估提示词模板构成的评估系统。目前此任务支持的数据集包括：

- openai_humaneval_with_readability

In [ ]:
dataset_name = "openai_humaneval_with_readability"
df = download_benchmark_dataset(task="code-readability-classification", dataset_name=dataset_name)
df.head()

## 展示可读性二元分类模板

查看用于可读性分类的默认模板。你可以调整模板，并与默认模板比较性能。

In [ ]:
CODE_READABILITY_PROMPT_TEMPLATE = """
你是一位严格但务实的高级软件工程师，非常关注代码的简单性与可读性。
请审核另一位工程师编写的以下代码，重点关注可读性。
如果代码可读，回复 "readable"；如果不可读，或相对于其目标显得不必要地复杂，
回复 "unreadable"。

只能回复 "readable" 或 "unreadable"

任务说明：
```
{input}
```

待评估实现：
```
{output}
```
"""

CODE_READABILITY_PROMPT_RAILS_MAP = {"readable": 1, "unreadable": 0}

模板变量：

- **input：**用户描述编码任务的查询
- **output：**编码任务的实现

## 配置 LLM

配置 OpenAI API 密钥。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
openai.api_key = openai_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

实例化 LLM 并设置参数。

## 基准数据集采样

样本量决定运行时间。
建议先用 100 个样本进行小规模迭代，再扩大测试集。

In [ ]:
df = df.sample(n=N_EVAL_SAMPLE_SIZE).reset_index(drop=True)
df = df.rename(
    columns={"prompt": "input", "solution": "output"},
)

## LLM 评估：使用 GPT-4 进行代码可读性分类

对部分数据运行可读性分类。

In [ ]:
model = LLM(provider="openai", model="gpt-4")

In [ ]:
code_readability_eval = ClassificationEvaluator(
    name="code readability",
    prompt_template=CODE_READABILITY_PROMPT_TEMPLATE,
    llm=model,
    choices=CODE_READABILITY_PROMPT_RAILS_MAP,
)

readability_classifications = await async_evaluate_dataframe(
    dataframe=df, evaluators=[code_readability_eval], concurrency=10
)

readability_classifications = readability_classifications["code readability_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

readability_classifications.tolist()


根据人工标注的真实可读性标签评估预测。

In [ ]:
true_labels = df["readable"].map(CODE_READABILITY_PROMPT_RAILS_MAP).tolist()

print(
    classification_report(
        true_labels, readability_classifications, labels=["readable", "unreadable"]
    )
)
confusion_matrix = ConfusionMatrix(
    actual_vector=true_labels,
    predict_vector=readability_classifications,
    classes=["readable", "unreadable"],
)
confusion_matrix.plot(
    cmap=plt.colormaps["Blues"],
    number_label=True,
    normalized=True,
)

## 检查评估结果

由于评估采用二元分类，我们可以轻松抽取几行与真实标签不一致的结果，查看对应的实际代码。

In [ ]:
df["readability"] = readability_classifications
# 检查真实标签为可读、但 LLM 判定为不可读的示例
filtered_df = df.query('readable == False and readability == "readable"')

# 检查满足条件的前 5 行
result = filtered_df.head(5)
result

## LLM 评估：使用 GPT-3.5 进行代码可读性分类

对部分数据运行可读性分类。

In [ ]:
model = LLM(provider="openai", model="gpt-3.5-turbo")

code_readability_eval = ClassificationEvaluator(
    name="code readability",
    prompt_template=CODE_READABILITY_PROMPT_TEMPLATE,
    llm=model,
    choices=CODE_READABILITY_PROMPT_RAILS_MAP,
)

readability_classifications = await async_evaluate_dataframe(
    dataframe=df, evaluators=[code_readability_eval], concurrency=10
)

readability_classifications = readability_classifications["code readability_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

readability_classifications.tolist()

In [ ]:
true_labels = df["readable"].map(CODE_READABILITY_PROMPT_RAILS_MAP).tolist()

print(
    classification_report(
        true_labels, readability_classifications, labels=["readable", "unreadable"]
    )
)
confusion_matrix = ConfusionMatrix(
    actual_vector=true_labels,
    predict_vector=readability_classifications,
    classes=["readable", "unreadable"],
)
confusion_matrix.plot(
    cmap=plt.colormaps["Blues"],
    number_label=True,
    normalized=True,
)

# 预览：GPT-4 Turbo

In [ ]:
model = LLM(provider="openai", model="gpt-4-turbo-preview")

code_readability_eval = ClassificationEvaluator(
    name="code readability",
    prompt_template=CODE_READABILITY_PROMPT_TEMPLATE,
    llm=model,
    choices=CODE_READABILITY_PROMPT_RAILS_MAP,
)

readability_classifications = await async_evaluate_dataframe(
    dataframe=df, evaluators=[code_readability_eval], concurrency=10
)

readability_classifications = readability_classifications["code readability_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

readability_classifications.tolist()

In [ ]:
true_labels = df["readable"].map(CODE_READABILITY_PROMPT_RAILS_MAP).tolist()

print(
    classification_report(
        true_labels, readability_classifications, labels=["readable", "unreadable"]
    )
)
confusion_matrix = ConfusionMatrix(
    actual_vector=true_labels,
    predict_vector=readability_classifications,
    classes=["readable", "unreadable"],
)
confusion_matrix.plot(
    cmap=plt.colormaps["Blues"],
    number_label=True,
    normalized=True,
)